# Observer.ai: eval dataset preparation

Builds `labeled_traces.jsonl`, the input to the Kev fine-tuning notebook.

**Flow**
1. Load HotpotQA (distractor) and sample questions
2. Plan runs: every question gets a natural run, some also get a seeded failure
3. Run the ReAct agent (`observer_agent` package) with resume support
4. Auto-label against gold answers (HotpotQA exact match + token F1)
5. Inspect label quality by condition
6. Split by question ID and export


In [1]:
import csv, hashlib, json, random, re, string, time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import replace
from pathlib import Path

import pandas as pd
from datasets import load_dataset
from tqdm.auto import tqdm

from observer_agent import AgentConfig, run_agent

# ---- config ----
N_QUESTIONS   = 3000            # smoke run first, then ~3000
SEED_FRAC     = 0.5           # share of questions that also get a seeded-failure run
CONDITION_MIX = {             # relative weights among seeded runs
    "missing_evidence": 1,
    "contradiction":    1,
    "early_stop":       1,
    "weak_model":       1,
}
HELDOUT_FRAC  = 0.2
WORKERS       = 4             # match OLLAMA_NUM_PARALLEL
SEED          = 7

MAIN_MODEL = "qwen2.5:7b"
WEAK_MODEL = "llama3.2:3b"
BASE_CFG = AgentConfig(
    model=MAIN_MODEL,
    base_url="http://localhost:11434/v1",
    max_tool_calls=6,
    max_steps=10,
    otel_endpoint=None,       # e.g. "http://localhost:6006/v1/traces" to also send traces to Phoenix
)

OUT_DIR    = Path("data/eval").resolve()
RAW_PATH   = OUT_DIR / "raw_runs.jsonl"
ERR_PATH   = OUT_DIR / "run_errors.jsonl"
FINAL_PATH = OUT_DIR / "labeled_traces.jsonl"
HAND_PATH  = OUT_DIR / "handcheck.csv"

OUT_DIR.mkdir(parents=True, exist_ok=True)
rng = random.Random(SEED)

## 1. Load HotpotQA (distractor)

Each question ships with 10 paragraphs: 2 gold (needed for the answer) and 8 distractors. The agent's tools search only these 10, so no web access is needed.

In [2]:
ds = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation")
print(ds)

def to_question(ex):
    paragraphs = [
        {"title": t, "text": " ".join(s.strip() for s in sents)}
        for t, sents in zip(ex["context"]["title"], ex["context"]["sentences"])
    ]
    return {
        "qid": ex["id"],
        "question": ex["question"],
        "answer": ex["answer"],
        "qtype": ex["type"],          # bridge or comparison
        "level": ex["level"],
        "paragraphs": paragraphs,
        "gold_titles": sorted(set(ex["supporting_facts"]["title"])),
    }

idx = rng.sample(range(len(ds)), N_QUESTIONS)
questions = [to_question(ds[i]) for i in idx]
print(f"{len(questions)} questions:", Counter(q["qtype"] for q in questions))

Dataset({
    features: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'],
    num_rows: 7405
})
3000 questions: Counter({'bridge': 2372, 'comparison': 628})


## 2. Plan runs and seed failures

| Condition | What changes | Failure it simulates |
|---|---|---|
| `natural` | nothing | whatever the model gets wrong on its own |
| `missing_evidence` | one gold paragraph removed | retrieval miss, agent must guess |
| `contradiction` | a decoy copy of the answer paragraph with a different answer is added | conflicting or poisoned sources |
| `early_stop` | tool budget of 1 | agent gives up before gathering enough |
| `weak_model` | smaller model | a cheaper model swapped in |

`contradiction` needs the gold answer to appear in a gold paragraph and not be yes/no. When it can't apply, the run falls back to `missing_evidence` and records that.

In [3]:
YES_NO = {"yes", "no"}

def plan_runs(questions):
    seeded = list(CONDITION_MIX)
    weights = [CONDITION_MIX[c] for c in seeded]
    runs = []
    for q in questions:
        runs.append((q, "natural"))
        if rng.random() < SEED_FRAC:
            runs.append((q, rng.choices(seeded, weights)[0]))
    return runs

def decoy_answer(q, pool, r):
    candidates = [p["answer"] for p in pool
                  if p["qid"] != q["qid"] and p["answer"].lower() not in YES_NO
                  and p["answer"].lower() != q["answer"].lower()]
    return r.choice(candidates) if candidates else None

def apply_condition(q, condition, pool):
    """Returns (paragraphs, cfg, applied_condition). Deterministic per (question, condition)."""
    r = random.Random(run_key(q, condition))
    paragraphs = [dict(p) for p in q["paragraphs"]]
    cfg = BASE_CFG

    if condition == "contradiction":
        ans = q["answer"]
        holder = next((p for p in paragraphs
                       if p["title"] in q["gold_titles"] and ans.lower() in p["text"].lower()), None)
        decoy = decoy_answer(q, pool, r)
        if ans.lower() in YES_NO or holder is None or decoy is None:
            condition = "missing_evidence"
        else:
            fake_text = re.sub(re.escape(ans), decoy, holder["text"], flags=re.IGNORECASE)
            paragraphs.insert(r.randrange(len(paragraphs) + 1),
                              {"title": f"{holder['title']} (archived revision)", "text": fake_text})
            return paragraphs, cfg, "contradiction"

    if condition == "missing_evidence":
        drop = r.choice(q["gold_titles"])
        paragraphs = [p for p in paragraphs if p["title"] != drop]
    elif condition == "early_stop":
        cfg = replace(cfg, max_tool_calls=1)
    elif condition == "weak_model":
        cfg = replace(cfg, model=WEAK_MODEL)

    return paragraphs, cfg, condition

def run_key(q, condition):
    return hashlib.sha256(f"{q['qid']}|{condition}".encode()).hexdigest()[:16]

runs = plan_runs(questions)
print(f"{len(runs)} planned runs:", Counter(c for _, c in runs))

4491 planned runs: Counter({'natural': 3000, 'contradiction': 398, 'missing_evidence': 372, 'early_stop': 368, 'weak_model': 353})


## 3. Run the agent

Runs are written to `raw_runs.jsonl` as they finish. Re-running this cell skips finished runs, so a crash or a closed laptop loses nothing.

Infrastructure errors (Ollama down, timeouts) go to `run_errors.jsonl` and are retried on the next run. They are never labeled as wrong answers.

In [4]:
def load_jsonl(path):
    if not path.exists():
        return []
    return [json.loads(l) for l in path.read_text().splitlines() if l.strip()]

def append_jsonl(path, row):
    with path.open("a") as f:
        f.write(json.dumps(row) + "\n")

def execute(q, condition):
    paragraphs, cfg, applied = apply_condition(q, condition, questions)
    result = run_agent(q["question"], paragraphs, cfg)
    return {
        "run_key": run_key(q, condition),
        "qid": q["qid"],
        "question": q["question"],
        "gold_answer": q["answer"],
        "qtype": q["qtype"],
        "level": q["level"],
        "gold_titles": q["gold_titles"],
        "planned_condition": condition,
        "failure_type": applied,
        **result,
    }

def run_all(runs, workers=WORKERS):
    done = {r["run_key"] for r in load_jsonl(RAW_PATH)}
    todo = [(q, c) for q, c in runs if run_key(q, c) not in done]
    print(f"{len(done)} already done, {len(todo)} to run")
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = {pool.submit(execute, q, c): (q, c) for q, c in todo}
        for fut in tqdm(as_completed(futures), total=len(futures)):
            q, c = futures[fut]
            try:
                append_jsonl(RAW_PATH, fut.result())
            except Exception as e:
                append_jsonl(ERR_PATH, {"run_key": run_key(q, c), "qid": q["qid"],
                                        "condition": c, "error": repr(e), "at": time.time()})

# Smoke test: 3 runs, then inspect one
run_all(runs[:3], workers=1)
print(json.dumps(load_jsonl(RAW_PATH)[-1], indent=2)[:3000])

1700 already done, 0 to run


0it [00:00, ?it/s]

{
  "run_key": "2bfc598f2e4f5964",
  "qid": "5ade4da055429939a52fe878",
  "question": "My Neighbor Totoro was produced by a Japanese animation film studio founded in what year?",
  "gold_answer": "1985",
  "qtype": "bridge",
  "level": "hard",
  "gold_titles": [
    "My Neighbor Totoro",
    "Studio Ghibli"
  ],
  "planned_condition": "missing_evidence",
  "failure_type": "missing_evidence",
  "trace_id": "85d064a3342a4ed1a0586f8c7648469b",
  "final_answer": "1985",
  "final_answer_raw": "1985",
  "tool_calls": [
    {
      "name": "search",
      "args": {
        "query": "Studio Ghibli founded"
      },
      "status": "ok",
      "result_preview": "[Short films by Studio Ghibli] Studio Ghibli is a Japanese animation film studio founded in 1985. In addition to producing 18 feature films, the studio has produced several short films, including commercials, films for the Ghibli Museum, music videos, and works released directly to video.\n\n[Ghibli M"
    }
  ],
  "evidence": [
    "St

In [5]:
# Full run
t0 = time.time()
run_all(runs)
print(f"took {(time.time() - t0) / 60:.1f} min; errors logged: {len(load_jsonl(ERR_PATH))}")

1700 already done, 2799 to run


  0%|          | 0/2799 [00:00<?, ?it/s]

/Users/samuelabolo/kode/hobby/observer-ai/finetune/.venv/lib/python3.13/site-packages/google/adk/models/llm_request.py:306: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  declaration = tool._get_declaration()
Task was destroyed but it is pending!
task: <Task pending name='Task-102' coro=<LoggingWorker._worker_loop() done, defined at /Users/samuelabolo/kode/hobby/observer-ai/finetune/.venv/lib/python3.13/site-packages/litellm/litellm_core_utils/logging_worker.py:221> wait_for=<Future cancelled>>
Task was destroyed but it is pending!
task: <Task pending name='Task-144' coro=<LoggingWorker._worker_loop() done, defined at /Users/samuelabolo/kode/hobby/observer-ai/finetune/.venv/lib/python3.13/site-packages/litellm/litellm_core_utils/logging_worker.py:221> wait_for=<Future cancelled>>
Task was destroyed but it is pending!
task: <Task pending name='Task-174' coro=<LoggingWorker._worker_loop() done, defined at /Users/samuelabolo/kode/hobby/observer-ai/

took 98.5 min; errors logged: 2


## 4. Auto-label against gold answers

Uses HotpotQA's official answer normalization and token F1 (a yes/no mismatch scores 0).

Starting thresholds, to be tuned in step 6:
- **correct:** exact match, or F1 ≥ 0.8
- **partial:** 0.3 ≤ F1 < 0.8
- **wrong:** F1 < 0.3, or no final answer

In [7]:
CORRECT_F1 = 0.8
PARTIAL_F1 = 0.3

def normalize_answer(s):
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def f1_score(pred, gold):
    p, g = normalize_answer(pred), normalize_answer(gold)
    if (p in {"yes", "no", "noanswer"} or g in {"yes", "no", "noanswer"}) and p != g:
        return 0.0
    pt, gt = p.split(), g.split()
    common = Counter(pt) & Counter(gt)
    same = sum(common.values())
    if same == 0:
        return 0.0
    precision, recall = same / len(pt), same / len(gt)
    return 2 * precision * recall / (precision + recall)

def label_run(r, correct_f1=CORRECT_F1, partial_f1=PARTIAL_F1):
    pred = (r.get("final_answer") or "").strip()
    if not pred:
        return "wrong", 0.0, False
    em = normalize_answer(pred) == normalize_answer(r["gold_answer"])
    f1 = f1_score(pred, r["gold_answer"])
    if em or f1 >= correct_f1:
        return "correct", f1, em
    if f1 >= partial_f1:
        return "partial", f1, em
    return "wrong", f1, em

raw = load_jsonl(RAW_PATH)
for r in raw:
    r["label"], r["f1"], r["em"] = label_run(r)
    r["gold_in_pred"] = normalize_answer(r["gold_answer"]) in normalize_answer(r.get("final_answer") or "")

df = pd.DataFrame(raw)
print(df["label"].value_counts())

label
wrong      2145
correct    1729
partial     625
Name: count, dtype: int64


## 5. Inspect the dataset

What to look for:
- **Base rate of wrong + partial** around 10 to 30%. Too low: raise `SEED_FRAC`. Too high: the natural runs are too weak.
- **Seeded conditions should fail more than `natural`.** If one doesn't, it isn't seeding anything.
- **`used_search` = False** means the model answered from memory. HotpotQA may be in its training data, so watch this share.
- **`gold_in_pred` with label `wrong`** usually means a verbose answer that F1 punished. Check these in the hand-check.

In [8]:
df["is_wrong"] = df["label"] != "correct"
print(f"base rate (wrong + partial): {df['is_wrong'].mean():.1%}\n")

print("label by condition:")
display(pd.crosstab(df["failure_type"], df["label"], normalize="index").round(2))

print("\nanswered without any tool call, by condition:")
display((~df["used_search"]).groupby(df["failure_type"]).mean().round(2))

print("\nstop reasons:")
display(df["stop_reason"].value_counts())

suspect = df[(df["label"] == "wrong") & df["gold_in_pred"]]
print(f"\n{len(suspect)} runs labeled wrong although the gold answer appears in the prediction")

df["approx_tokens"] = df.apply(
    lambda r: (len(r["question"]) + len(r.get("final_answer") or "") + sum(len(e) for e in r["evidence"])) / 4, axis=1)
print(f"approx slice tokens: median {df['approx_tokens'].median():.0f}, p95 {df['approx_tokens'].quantile(0.95):.0f}")

base rate (wrong + partial): 61.6%

label by condition:


label,correct,partial,wrong
failure_type,,,
contradiction,0.40,0.14,0.46
early_stop,0.39,0.15,0.46
missing_evidence,0.32,0.09,0.59
natural,0.41,0.14,0.45
weak_model,0.19,0.21,0.60



answered without any tool call, by condition:


failure_type
contradiction       0.04
early_stop          0.05
missing_evidence    0.08
natural             0.06
weak_model          0.03
Name: used_search, dtype: float64


stop reasons:


stop_reason
answered     4477
no_answer      22
Name: count, dtype: int64


186 runs labeled wrong although the gold answer appears in the prediction
approx slice tokens: median 369, p95 710


## 6. Split by question ID and export

Every run of a question (natural and seeded) lands in the same split, so the held-out set never contains a question the model trained on.

The output keeps the fields the fine-tuning notebook needs (`trace_id`, `question`, `tool_calls`, `evidence`, `final_answer`, `label`) plus `split` and analysis extras.

In [9]:
qids = sorted(df["qid"].unique())
rng_split = random.Random(SEED)
rng_split.shuffle(qids)
heldout_qids = set(qids[: int(len(qids) * HELDOUT_FRAC)])
df["split"] = df["qid"].map(lambda q: "heldout" if q in heldout_qids else "train")

assert not set(df.loc[df["split"] == "train", "qid"]) & set(df.loc[df["split"] == "heldout", "qid"])

KEEP = ["trace_id", "question", "tool_calls", "evidence", "final_answer", "label", "split",
        "qid", "gold_answer", "f1", "em", "failure_type", "qtype", "level", "model",
        "used_search", "stop_reason", "steps", "latency_ms"]

with FINAL_PATH.open("w") as f:
    for row in df[KEEP].to_dict(orient="records"):
        f.write(json.dumps(row, default=str) + "\n")

summary = df.groupby("split").agg(traces=("trace_id", "size"), questions=("qid", "nunique"),
                                   wrong_rate=("is_wrong", "mean"))
display(summary.round(3))
print(f"wrote {len(df)} traces to {FINAL_PATH}")

,traces,questions,wrong_rate
split,,,
heldout,900,600,0.60
train,3599,2400,0.62


wrote 4499 traces to /Users/samuelabolo/kode/hobby/observer-ai/finetune/notebooks/data/eval/labeled_traces.jsonl
